# Figure 1 and Supplementary Figure 1: cohort overview

**Inputs** (input_dir)
- `metadata/sample_metadata.tsv`: 91 cfRNA samples (cohort, timepoint, PFS, read depth, extraction and sequencing batch)
- `counts/counts_TMMCPM_cart_paper_ALLDATA.tsv.gz`: TMM-CPM matrix; only the 91 cfRNA columns are used

**Outputs** (plots/)
- `Fig1B_pfs_by_cohort.pdf`: PFS Kaplan-Meier curves by cohort
- `Fig1C_read_depth_features.pdf`: total reads and feature counts per cohort
- `Fig1D_pca_by_cohort.pdf`: PCA of the top 500 variable transcripts (colour = cohort, shape = timepoint)
- `SFig1A_pca_by_sequencing_batch.pdf`: the same PCA coloured by sequencing batch
- `SFig1B_pca_by_extraction_batch.pdf` and `SFig1B_pca_by_extraction_batch_legend.pdf`: the same PCA coloured by extraction group, shape = extraction year

Fig1A (the cohort schematic) was not made in code.

In [ ]:
source("../../utils/setup.R")
library(survminer)

## Load data

In [ ]:
mdf <- load_sample_metadata()

# TMM-CPM of the 91 cfRNA samples (the PBMC and TME columns are dropped)
counts_tmmcpm <- load_counts("tmmcpm_all")[, mdf$sample_id]

## Figure 1B: PFS by cohort

Patients infused before 2024-12-01 (`km_include`). `pfs_time_km_days` sets censored patients to 500 days, past the 400-day window.

In [ ]:
km_cohort <- km_by_group(filter(mdf, km_include), group = "cohort",
                         palette = COHORT_COLORS, major_grid = TRUE)
km_cohort
save_plot(km_cohort, "Fig1B_pfs_by_cohort", width = 2.145, height = 2.145)

## Figure 1C: sequencing depth and feature counts

In [ ]:
depth_violin <- function(y, y_lab) {
    ggplot(mdf, aes(x = cohort, y = .data[[y]], fill = cohort)) +
        geom_violin(trim = FALSE, alpha = 0.6) +
        geom_boxplot(width = 0.12, outlier.shape = NA, alpha = 0.8) +
        geom_point(aes(color = cohort), size = 1.25, alpha = 0.75, stroke = NA,
                   position = position_jitter(width = 0.3, height = 0, seed = 42)) +
        scale_color_manual(values = COHORT_COLORS) +
        scale_fill_manual(values = COHORT_COLORS) +
        scale_y_log10() +
        labs(y = y_lab) +
        theme_cfrna_print() +
        theme(axis.title.x = element_blank()) +
        theme_axis_lines()
}

depth_plt <- depth_violin("reads_all", "Total Reads") + depth_violin("ftcount", "Feature Counts") +
    plot_layout(ncol = 2)
depth_plt
save_plot(depth_plt, "Fig1C_read_depth_features", width = 4.3, height = 2)

## PCA of the top 500 variable transcripts

Computed once and used for Fig1D and SFig1. The legend reports PC1 25% and PC2 13.9%.

In [ ]:
gene_var <- apply(counts_tmmcpm, 1, var)
top_500 <- names(sort(gene_var, decreasing = TRUE)[1:500])
pca_res <- prcomp(t(counts_tmmcpm[top_500, ]), scale. = TRUE)
pca_var_pct <- round(100 * pca_res$sdev^2 / sum(pca_res$sdev^2), 1)
print(pca_var_pct[1:2])

pca_df <- data.frame(sample_id = rownames(pca_res$x), PC1 = pca_res$x[, 1], PC2 = pca_res$x[, 2]) %>%
    left_join(mdf, by = "sample_id") %>%
    mutate(extraction_year = factor(extraction_year),
           extraction_group = factor(extraction_group))

pca_theme <- theme_cfrna_print() +
    theme(plot.title = element_blank(),
          panel.grid.major = element_line(color = "grey95", linewidth = 0.5))
pca_labs <- labs(x = paste0("PC1 (", pca_var_pct[1], "%)"),
                 y = paste0("PC2 (", pca_var_pct[2], "%)"))

## Figure 1D: PCA by cohort and timepoint

The x axis is limited to c(-35, 30), so all 91 samples are shown (the leftmost has PC1 = -32.1). The axes are labelled "PC1"/"PC2"; the variance explained is given in the legend.

In [ ]:
pca_cohort <- ggplot(pca_df, aes(x = PC1, y = PC2, color = cohort, shape = timepoint)) +
    geom_point() +
    pca_theme +
    # c(-35, 30) keeps the sample at PC1 = -32.1, which c(-30, 30) would drop
    scale_x_continuous(limits = c(-35, 30), breaks = seq(-30, 30, 10)) +
    scale_color_manual(values = COHORT_COLORS) +
    labs(x = "PC1", y = "PC2")
pca_cohort
save_plot(pca_cohort, "Fig1D_pca_by_cohort", width = 2.145, height = 1.5)

## Supplementary Figure 1A: PCA by sequencing batch

In [ ]:
pca_batch <- ggplot(pca_df, aes(x = PC1, y = PC2, color = sequencing_batch)) +
    geom_point(size = 1) +
    pca_theme +
    theme(legend.position = "left") +
    scale_color_manual(values = setNames(COHORT_COLORS, paste("Batch", 1:3)),
                       name = "Sequencing batch") +
    pca_labs
pca_batch
save_plot(pca_batch, "SFig1A_pca_by_sequencing_batch", width = 4, height = 2)

## Supplementary Figure 1B: PCA by extraction batch

Colour = extraction group, shape = extraction year (the full extraction date is not distributed). The legend is saved as a separate file.

In [ ]:
pca_extraction <- ggplot(pca_df, aes(x = PC1, y = PC2, color = extraction_group, shape = extraction_year)) +
    geom_point(size = 1) +
    pca_theme +
    theme(legend.position = "left") +
    scale_shape_manual(values = c("2024" = 16, "2025" = 17)) +
    pca_labs
pca_extraction

save_plot(ggpubr::as_ggplot(ggpubr::get_legend(pca_extraction)),
          "SFig1B_pca_by_extraction_batch_legend", width = 4, height = 5)
save_plot(pca_extraction + theme(legend.position = "none"),
          "SFig1B_pca_by_extraction_batch", width = 4, height = 5)